# Pluto M1 on Kaggle: first real model (~6M params, TinyStories)

Settings → **Accelerator: GPU (T4 ×2 or P100)**, **Internet: on** (needs a phone-verified Kaggle account).

Add-ons → Secrets (never paste tokens into cells):
- `HF_TOKEN` (**recommended**): a Hugging Face *write* token. Checkpoints and logs are synced to a
  private Hub repo, so the run survives the session being killed.
- `GITHUB_TOKEN`: only needed while the GitHub repo is private.

In [ ]:
import os, subprocess
REPO = "thedeepakreddy/Pluto-LLM"
BRANCH = "main"
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

def secret(name):
    try:
        return secrets.get_secret(name) if secrets else os.environ.get(name)
    except Exception:
        return None

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
if not os.path.exists("Pluto-LLM"):
    subprocess.run(["git", "clone", "-b", BRANCH, url, "Pluto-LLM"], check=True)
%cd Pluto-LLM
!pip install -q -e ".[hub]"

hf = secret("HF_TOKEN")
HUB = ""
if hf:
    os.environ["HF_TOKEN"] = hf
    from huggingface_hub import HfApi
    HUB = HfApi().whoami()["name"] + "/pluto-runs"   # private repo, created on first upload
print("Hub sync:", HUB or "OFF (set the HF_TOKEN secret to survive session kills)")
SYNC = f"--set train.hub_repo={HUB}" if HUB else ""

## 1. GPU check and fp16 smoke test

In [ ]:
import torch
from llm.train.loop import resolve_device, resolve_precision
dev = resolve_device("auto")
print(torch.__version__, dev, torch.cuda.get_device_name(0) if dev.type == "cuda" else "")
print("auto precision ->", resolve_precision("auto", dev))   # T4/P100 -> fp16, A100/H100 -> bf16

In [ ]:
# Tiny overfit run with fp16 + GradScaler forced: proves the free-GPU precision path works.
!llm prepare-data configs/m1-overfit.yaml
!llm train-tokenizer configs/m1-overfit.yaml
!llm tokenize configs/m1-overfit.yaml
!llm train configs/m1-overfit.yaml --set train.device=cuda --set train.precision=fp16 --set train.max_steps=300

## 2. Prepare TinyStories (download, clean, tokenizer, shards)
CPU-bound; expect roughly 15–40 minutes. Re-running it gives identical artifacts.

In [ ]:
CFG = "configs/m1-tinystories.yaml"
!llm prepare-data {CFG}
!cat data/tinystories-v1/data_report.md
!llm train-tokenizer {CFG}
!llm tokenize {CFG}

## 3. Train, with a deliberate interruption at step 500

In [ ]:
!llm train {CFG} {SYNC} --stop-at-step 500
import glob
RUN = sorted(glob.glob("runs/m1-tinystories-*"))[-1]
RUN_ID = RUN.split("/")[-1]
print(RUN_ID)
!tail -3 {RUN}/metrics.jsonl

## 4. Resume to the end

**If the session died:** run the install cell, then section 2 (rebuilds identical data/tokenizer/shards),
set `RUN_ID` below to the printed id, and run this cell. It pulls the run from the Hub and continues
from the last checkpoint.

In [ ]:
# RUN_ID = "m1-tinystories-..."   # only needed after a session restart
RUN = f"runs/{RUN_ID}"
import os
if not os.path.exists(RUN) and HUB:
    !llm pull-run {HUB} {RUN_ID}
!llm resume {RUN}

## 5. Generate and report

In [ ]:
!llm generate {RUN} --prompt "Once upon a time, there was a little dog named" --seed 1
!llm generate {RUN} --prompt "Lily wanted to go to the park, but" --seed 1

In [ ]:
# Write honest limitations before sharing, then build the report.
open(f"{RUN}/limitations.md", "w").write(
    "# Known limitations\n\n"
    "- ~6M parameters trained on synthetic TinyStories only: simple children's-story English.\n"
    "- No instruction tuning; it continues text rather than answering questions.\n"
    "- Single run, single seed; no ablations yet.\n")
!llm report {RUN}
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN}/report.md").read().replace("](charts/", f"]({RUN}/charts/")))